# Benchmark — Silver subjects transformation

## Benchmark and identify improvement
Measure ONE representative transformation and review the Spark plan for a practical improvement.

## Representative transformation chosen
Notebook `Clinical Trial Silver Transformation`, cell 6 **Transform: subjects** — bronze read → trim/casts → valid/invalid split on `site_id` → `dropDuplicates(["subject_id"])` → MERGE upserts → counts. It is the heaviest Silver step (dedup + quarantine + 5 column casts).

## Baseline result (serverless, read-only replica — silver tables untouched)
- Runs (after 1 warm-up): **1.34s / 1.00s / 1.00s → median 1.00s, mean 1.11s**
- Correctness guardrails: valid deduped = **267**, quarantined = **5** (assertions match Silver)

## Optimization idea (evidence-based, see plan review cell)
Cell 6 fires **4 actions** over the same lineage (MERGE subjects, `deduped.count()` AFTER the MERGE, MERGE quarantine, `invalid.count()`), so the bronze scan + casts + dedup aggregation is recomputed up to 4×.

## 29-Oct result (applied optimization: MERGE-metrics counts)
- **Change implemented:** the two post-MERGE `count()` actions were replaced with counts read from the MERGE's own `DESCRIBE HISTORY` row metrics (`numTargetRowsInserted` + `numTargetRowsUpdated`) — O(1) metadata read instead of an O(data) bronze re-scan; cell actions 4 → 2.
- **Same-data timing (full cell-6 replica incl. real MERGEs, warm-up + 3 runs):** BEFORE median **11.65s** (4 actions) vs AFTER median **12.24s** (2 actions) → 0.95x, within run-to-run noise; wall-clock at 280-row scale is dominated by Delta MERGE commit overhead. Speedup not mandatory per acceptance; the removed jobs are full scans that scale with data size.
- **Cache variant rejected by measurement:** `df.cache()` smoke test was 5.97s vs 1.00s baseline read-only — caching overhead exceeds scan savings at this scale.
- **Correctness:** 267 valid deduped / 5 quarantined on every run (asserted); regression checks vs Gold KPIs documented in the 29-Oct verification.

In [0]:
# --- Benchmark harness: Silver subjects transformation (read-only) ---
# Faithful replica of Silver notebook cell 6 WITHOUT the MERGE writes.
# USE_CACHE=False = current production behavior (baseline, 28-Oct).
# USE_CACHE=True  = candidate optimization for 29-Oct (cache deduped + invalid
#                   so the 4 cell-6 actions reuse ONE bronze scan instead of 4).

import time, datetime, statistics
from pyspark.sql.functions import col, trim, to_date, lit

USE_CACHE = False  # 28-Oct baseline; flip to True for the 29-Oct before/after timing
BRONZE_SCHEMA = "workspace.bronze"
SILVER_TS = datetime.datetime.now(datetime.UTC).isoformat()

def subjects_transform(use_cache=False):
    df_subjects_raw = spark.table(f"{BRONZE_SCHEMA}.subjects") \
        .select(
            col("subject_id"),
            col("study_id"),
            trim(col("site_id")).alias("site_id"),
            to_date(col("screening_date"), "yyyy-MM-dd").alias("screening_date"),
            to_date(col("enrollment_date"), "yyyy-MM-dd").alias("enrollment_date"),
            col("birth_year").cast("int").alias("birth_year"),
            col("Sex").alias("sex"),
            col("treatment_arm"),
            col("subject_status"),
            lit(SILVER_TS).alias("silver_load_timestamp")
        )
    df_valid = df_subjects_raw.filter(col("site_id").isNotNull() & (trim(col("site_id")) != ""))
    df_invalid = df_subjects_raw.filter(col("site_id").isNull() | (trim(col("site_id")) == ""))
    df_deduped = df_valid.dropDuplicates(["subject_id"])

    if use_cache:
        df_deduped = df_deduped.cache()
        df_invalid = df_invalid.cache()
        df_deduped.count()      # materialize once here
        df_invalid.count()

    # Terminal actions mirroring the notebook's counts
    n_valid = df_deduped.count()
    n_quar = df_invalid.count()
    if use_cache:
        df_deduped.unpersist(); df_invalid.unpersist()
    return n_valid, n_quar

# Warm-up (discarded)
n_v, n_q = subjects_transform(use_cache=USE_CACHE)
print(f"warm-up: valid_deduped={n_v}, quarantined={n_q}")

# 3 timed runs
label = "OPTIMIZED (cache)" if USE_CACHE else "BASELINE"
timings = []
for i in range(3):
    t0 = time.perf_counter()
    n_valid, n_quar = subjects_transform(use_cache=USE_CACHE)
    timings.append(time.perf_counter() - t0)
    print(f"{label} run {i+1}: {timings[-1]:.2f}s  (valid_deduped={n_valid}, quarantined={n_quar})")

print(f"\n{label} subjects transform: runs={[f'{t:.2f}s' for t in timings]}  "
      f"median={statistics.median(timings):.2f}s  mean={statistics.mean(timings):.2f}s")

# Correctness guardrails (must match Silver results on every run)
assert n_valid == 267, f"expected 267 valid deduped, got {n_valid}"
assert n_quar == 5, f"expected 5 quarantined, got {n_quar}"
print("Guardrails passed: 267 valid deduped / 5 quarantined (matches Silver)")

In [0]:
# --- Plan review: EXPLAIN FORMATTED for the dedup lineage ---
# Serverless-safe (no JVM access): plans are returned as SQL text.

from pyspark.sql.functions import col, trim, to_date, lit
import datetime

SILVER_TS = datetime.datetime.now(datetime.UTC).isoformat()

df_subjects_raw = spark.table("workspace.bronze.subjects") \
    .select(
        col("subject_id"), col("study_id"),
        trim(col("site_id")).alias("site_id"),
        to_date(col("screening_date"), "yyyy-MM-dd").alias("screening_date"),
        to_date(col("enrollment_date"), "yyyy-MM-dd").alias("enrollment_date"),
        col("birth_year").cast("int").alias("birth_year"),
        col("Sex").alias("sex"), col("treatment_arm"), col("subject_status"),
        lit(SILVER_TS).alias("silver_load_timestamp")
    )
df_valid = df_subjects_raw.filter(col("site_id").isNotNull() & (trim(col("site_id")) != ""))
df_invalid = df_subjects_raw.filter(col("site_id").isNull() | (trim(col("site_id")) == ""))
df_deduped = df_valid.dropDuplicates(["subject_id"])

df_deduped.createOrReplaceTempView("bm_deduped")
df_invalid.createOrReplaceTempView("bm_invalid")

planA = spark.sql("EXPLAIN FORMATTED SELECT * FROM bm_deduped").collect()[0][0]
planB = spark.sql("EXPLAIN FORMATTED SELECT * FROM bm_invalid").collect()[0][0]

print("=" * 70)
print("PLAN A — valid + dropDuplicates(subject_id) [silver.subjects MERGE source]")
print("=" * 70)
print(planA)
print("=" * 70)
print("PLAN B — invalid (quarantine MERGE source)")
print("=" * 70)
print(planB)

# Plan review findings 

1. **One PhotonScan of `workspace.bronze.subjects` per action, no caching.** Cell 6 performs 4 actions over the same lineage: MERGE subjects → `df_subjects_deduped.count()` (AFTER the MERGE) → MERGE quarantine → `df_subjects_invalid.count()`. Each action re-runs the bronze scan, the casts, and the dedup aggregation → up to **4× recomputation** of identical work.
2. **Redundant `trim`.** The pushed-down predicate shows `trim(trim(site_id))` — `trim` is applied in the select and re-applied in the filter (`col("site_id").isNotNull() & (trim(col("site_id")) != "")` after the alias already trimmed it). Harmless but removable.
3. **`dropDuplicates(["subject_id"])` compiles to `PhotonGroupingAgg` with `first(...)` over all columns** grouped by `subject_id` — a correct, shuffle-free plan at this data size; no evidence it needs replacing.

**Evidence-based optimization (for 29-Oct):** the cell's biggest measured inefficiency is the **4 separate actions over the same lineage** — each one is a full bronze scan + cast + dedup job. Two candidate fixes, to be timed with the same harness on 29-Oct:

* **(a) Eliminate the two redundant `count()` actions** — the counts after each MERGE can be read from the MERGE's own row metrics (`num_target_rows_inserted` + `num_target_rows_updated`), removing 2 of the 4 scans/jobs with zero output change.
* **(b) Cache the deduped/invalid DataFrames** — validated in a smoke test (29 Oct: 267/5 correct), BUT at this 280-row scale the cache variant was **slower** (5.97s vs 1.00s baseline) because caching overhead exceeds the scan savings. Prefer (a); consider (b) only if timing shows a win.

Regression checks for either fix: 267 valid deduped / 5 quarantined, DQ checks pass, Gold KPI tables unchanged.

In [0]:
# --- 29-Oct: BEFORE/AFTER timing of the FULL cell-6 subjects transform (with real MERGEs) ---
# Same data, same harness. MERGEs are idempotent upserts on identical rows (same as the
# daily job), so silver tables end with the same 267/5 rows either way.
#
# BEFORE = original cell 6: 4 actions over the bronze lineage
#          (MERGE subjects -> deduped.count() AFTER merge -> MERGE quarantine -> invalid.count())
# AFTER  = 29-Oct optimization: 2 actions
#          (MERGE subjects -> count from DESCRIBE HISTORY metrics; same for quarantine)

import time, datetime, statistics
from pyspark.sql.functions import col, trim, to_date, lit

SILVER_SCHEMA = "workspace.silver"
BRONZE_SCHEMA = "workspace.bronze"

def build_lineage():
    """Exact cell-6 lineage: read bronze -> trim/casts -> valid/invalid split -> dedup."""
    silver_ts = datetime.datetime.now(datetime.UTC).isoformat()
    df_raw = spark.table(f"{BRONZE_SCHEMA}.subjects").select(
        col("subject_id"), col("study_id"),
        trim(col("site_id")).alias("site_id"),
        to_date(col("screening_date"), "yyyy-MM-dd").alias("screening_date"),
        to_date(col("enrollment_date"), "yyyy-MM-dd").alias("enrollment_date"),
        col("birth_year").cast("int").alias("birth_year"),
        col("Sex").alias("sex"), col("treatment_arm"), col("subject_status"),
        lit(silver_ts).alias("silver_load_timestamp")
    )
    df_valid = df_raw.filter(col("site_id").isNotNull() & (trim(col("site_id")) != ""))
    df_invalid = df_raw.filter(col("site_id").isNull() | (trim(col("site_id")) == ""))
    return df_valid.dropDuplicates(["subject_id"]), df_invalid

def run_before():
    """Original cell 6: 4 actions (2 MERGEs + 2 post-merge count() jobs)."""
    df_deduped, df_invalid = build_lineage()
    df_deduped.createOrReplaceTempView("temp_subjects")
    spark.sql(f"""
        MERGE INTO {SILVER_SCHEMA}.subjects AS target
        USING temp_subjects AS source
        ON target.subject_id = source.subject_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    n_valid = df_deduped.count()          # extra job: re-scans bronze + re-runs dedup
    df_invalid.createOrReplaceTempView("temp_subjects_q")
    spark.sql(f"""
        MERGE INTO {SILVER_SCHEMA}.subjects_quarantine AS target
        USING temp_subjects_q AS source
        ON target.subject_id = source.subject_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    n_quar = df_invalid.count()           # extra job: re-scans bronze
    return n_valid, n_quar

def merge_history_count(table):
    """Optimized count: read the MERGE's own row metrics from Delta history (no data scan)."""
    m = spark.sql(f"DESCRIBE HISTORY {SILVER_SCHEMA}.{table} LIMIT 1").collect()[0]["operationMetrics"]
    return int(m.get("numTargetRowsInserted", 0)) + int(m.get("numTargetRowsUpdated", 0))

def run_after():
    """Optimized cell 6: 2 actions (2 MERGEs; counts from history metrics)."""
    df_deduped, df_invalid = build_lineage()
    df_deduped.createOrReplaceTempView("temp_subjects")
    spark.sql(f"""
        MERGE INTO {SILVER_SCHEMA}.subjects AS target
        USING temp_subjects AS source
        ON target.subject_id = source.subject_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    n_valid = merge_history_count("subjects")
    df_invalid.createOrReplaceTempView("temp_subjects_q")
    spark.sql(f"""
        MERGE INTO {SILVER_SCHEMA}.subjects_quarantine AS target
        USING temp_subjects_q AS source
        ON target.subject_id = source.subject_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    n_quar = merge_history_count("subjects_quarantine")
    return n_valid, n_quar

def bench(fn, label, n_runs=3):
    n_v, n_q = fn()                      # warm-up (discarded)
    assert (n_v, n_q) == (267, 5), f"{label} warm-up guardrail failed: {n_v}/{n_q}"
    timings = []
    for i in range(n_runs):
        t0 = time.perf_counter()
        n_v, n_q = fn()
        timings.append(time.perf_counter() - t0)
        assert (n_v, n_q) == (267, 5), f"{label} run {i+1} guardrail failed: {n_v}/{n_q}"
        print(f"{label} run {i+1}: {timings[-1]:.2f}s  (valid_deduped={n_v}, quarantined={n_q})")
    print(f"{label}: median={statistics.median(timings):.2f}s  mean={statistics.mean(timings):.2f}s\n")
    return timings

print("== BEFORE (original cell 6, 4 actions) ==")
t_before = bench(run_before, "BEFORE")

print("== AFTER (optimized cell 6, 2 actions) ==")
t_after = bench(run_after, "AFTER")

mb, ma = statistics.median(t_before), statistics.median(t_after)
print(f"RESULT: BEFORE median={mb:.2f}s  AFTER median={ma:.2f}s  "
      f"speedup={mb/ma:.2f}x  (guardrails 267/5 passed on every run)")